# Looped Block-ELL Scaling Law Analysis

Loads completed grid search results from Drive and produces scaling law plots.
Run on **CPU runtime** — no TPU needed.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

!pip install -q wandb scipy

import wandb
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except:
    pass
wandb.login()

# Pull all completed runs from wandb (no Drive needed)
api = wandb.Api()
all_runs = api.runs("looped-blockell-scaling", filters={"state": "finished"})

all_results = []
for r in all_runs:
    cfg = r.config
    summ = r.summary
    val_ppl = summ.get("final_val_ppl") or summ.get("val/ppl")
    if val_ppl is None:
        continue
    all_results.append({
        "d_model": cfg.get("d_model"),
        "n_heads": cfg.get("n_heads"),
        "d_ff": cfg.get("d_ff"),
        "n_core": cfg.get("n_core"),
        "n_params": cfg.get("n_params", 0),
        "effective_depth": cfg.get("effective_depth"),
        "final_val_ppl": float(val_ppl),
        "final_density": summ.get("final_density", cfg.get("final_density", 1.0)),
        "final_train_loss": summ.get("train/loss", float("nan")),
    })

df = pd.DataFrame(all_results)
print(f"Loaded {len(df)} completed configs from wandb.\n")

GRID_ALL = [(d, nc) for d in [256, 384, 512, 768, 1024] for nc in [2, 4, 6, 8]]
completed = set(zip(df["d_model"].astype(int), df["n_core"].astype(int)))
missing = [g for g in GRID_ALL if g not in completed]
print("Missing configs:")
for d, nc in missing:
    print(f"  d={d}, n_core={nc}")
print()

# Also save locally for other cells
RESULTS_FILE = "scaling_results.json"
with open(RESULTS_FILE, "w") as f:
    json.dump(all_results, f, indent=2)


In [ ]:
# ─── Full results table ────────────────────────────────────────────────────────
df['tok_per_param'] = (77000 * 32 * 1024) / df['n_params']

df_sorted = df.sort_values('final_val_ppl').reset_index(drop=True)
print('All results sorted by val PPL (lower = better):')
print(df_sorted[[
    'd_model', 'n_core', 'n_params', 'effective_depth',
    'final_val_ppl', 'final_density', 'tok_per_param'
]].to_string(index=False, float_format=lambda x: f'{x:.2f}' if x < 1000 else f'{x:.0f}'))

best = df_sorted.iloc[0]
print(f'\nBest: d={int(best["d_model"])}, n_core={int(best["n_core"])}'
      f'  →  val_ppl={best["final_val_ppl"]:.2f}'
      f'  ({best["n_params"]/1e6:.1f}M params, eff_depth={int(best["effective_depth"])})')

In [ ]:
# ─── 2D Heatmap: d_model × n_core → val_ppl ──────────────────────────────────
d_vals  = sorted(df['d_model'].unique())
nc_vals = sorted(df['n_core'].unique())

ppl_matrix = np.full((len(nc_vals), len(d_vals)), np.nan)
for _, row in df.iterrows():
    ri = nc_vals.index(row['n_core'])
    ci = d_vals.index(row['d_model'])
    ppl_matrix[ri, ci] = row['final_val_ppl']

fig, ax = plt.subplots(figsize=(10, 6))
vmin = np.nanmin(ppl_matrix)
vmax = np.nanmax(ppl_matrix)
im = ax.imshow(ppl_matrix, aspect='auto', cmap='RdYlGn_r',
               vmin=vmin, vmax=vmax, origin='lower')
ax.set_xticks(range(len(d_vals)))
ax.set_xticklabels([str(int(d)) for d in d_vals], fontsize=12)
ax.set_yticks(range(len(nc_vals)))
ax.set_yticklabels([str(int(n)) for n in nc_vals], fontsize=12)
ax.set_xlabel('d_model (width)', fontsize=13)
ax.set_ylabel('n_core (unique layers)', fontsize=13)
ax.set_title('Val PPL: Width × Depth Grid (lower = better)', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Val PPL', shrink=0.8)

for ri in range(len(nc_vals)):
    for ci in range(len(d_vals)):
        v = ppl_matrix[ri, ci]
        if np.isnan(v):
            ax.text(ci, ri, 'OOM', ha='center', va='center',
                    fontsize=10, color='red', fontweight='bold')
        else:
            frac = (v - vmin) / (vmax - vmin + 1e-9)
            color = 'white' if frac > 0.6 or frac < 0.3 else 'black'
            ax.text(ci, ri, f'{v:.1f}', ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved heatmap.png')

In [ ]:
# ─── Scatter: params vs val_ppl, colored by n_core ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Looped Block-ELL Scaling Laws (Loop+Prune to 28%)',
             fontsize=14, fontweight='bold')

# Plot 1: Params vs PPL colored by n_core
ax = axes[0]
nc_unique = sorted(df['n_core'].unique())
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(nc_unique)))
for i, nc in enumerate(nc_unique):
    mask = df['n_core'] == nc
    sub = df[mask].sort_values('d_model')
    ax.plot(sub['n_params']/1e6, sub['final_val_ppl'], 'o-',
            color=colors[i], markersize=8, lw=2, label=f'n_core={int(nc)}')
    for _, row in sub.iterrows():
        ax.annotate(f'd{int(row["d_model"])}',
                    (row['n_params']/1e6, row['final_val_ppl']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('Parameters (M)', fontsize=12)
ax.set_ylabel('Val PPL', fontsize=12)
ax.set_title('Width Scaling at Fixed Core Depths', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Plot 2: Effective depth vs PPL colored by d_model
ax = axes[1]
d_unique = sorted(df['d_model'].unique())
colors2 = plt.cm.plasma(np.linspace(0.1, 0.9, len(d_unique)))
for i, d in enumerate(d_unique):
    mask = df['d_model'] == d
    sub = df[mask].sort_values('effective_depth')
    ax.plot(sub['effective_depth'], sub['final_val_ppl'], 's-',
            color=colors2[i], markersize=8, lw=2, label=f'd={int(d)}')
ax.set_xlabel('Effective Depth (1 + n_core×4 + 1)', fontsize=12)
ax.set_ylabel('Val PPL', fontsize=12)
ax.set_title('Depth Scaling at Fixed Widths', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('scaling_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved scaling_curves.png')

In [ ]:
# ─── Efficiency frontier: PPL per million params ─────────────────────────────
df['ppl_per_M'] = df['final_val_ppl'] / (df['n_params'] / 1e6)

fig, ax = plt.subplots(figsize=(10, 6))

# Pareto frontier
df_par = df.sort_values('n_params')
pareto_ppl = float('inf')
pareto_x, pareto_y = [], []
for _, row in df_par.iterrows():
    if row['final_val_ppl'] < pareto_ppl:
        pareto_ppl = row['final_val_ppl']
        pareto_x.append(row['n_params']/1e6)
        pareto_y.append(row['final_val_ppl'])

ax.plot(pareto_x, pareto_y, 'k--', lw=1.5, alpha=0.5, label='Pareto frontier')
ax.fill_between(pareto_x, pareto_y, max(df['final_val_ppl'])*1.1,
                alpha=0.05, color='green')

# All points
for nc in sorted(df['n_core'].unique()):
    mask = df['n_core'] == nc
    sub = df[mask]
    ax.scatter(sub['n_params']/1e6, sub['final_val_ppl'],
              s=100, zorder=3, label=f'n_core={int(nc)}',
              edgecolors='k', linewidths=0.5)
    for _, row in sub.iterrows():
        ax.annotate(f'd{int(row["d_model"])}',
                    (row['n_params']/1e6, row['final_val_ppl']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points')

ax.set_xlabel('Parameters (M)', fontsize=12)
ax.set_ylabel('Val PPL', fontsize=12)
ax.set_title('Efficiency Frontier: Looped+Pruned Transformer', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('efficiency_frontier.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved efficiency_frontier.png')

In [ ]:
# ─── Key insights ─────────────────────────────────────────────────────────────
print('='*60)
print('SCALING LAW INSIGHTS')
print('='*60)

# 1. Does more depth help at fixed width?
print('\n1. DEPTH SCALING (does more n_core help at fixed width?)')
for d in sorted(df['d_model'].unique()):
    sub = df[df['d_model'] == d].sort_values('n_core')
    if len(sub) >= 2:
        best = sub.loc[sub['final_val_ppl'].idxmin()]
        worst = sub.loc[sub['final_val_ppl'].idxmax()]
        improvement = (worst['final_val_ppl'] - best['final_val_ppl']) / worst['final_val_ppl'] * 100
        print(f'  d={int(d):>5}: best n_core={int(best["n_core"])} (PPL {best["final_val_ppl"]:.1f}), '
              f'worst n_core={int(worst["n_core"])} (PPL {worst["final_val_ppl"]:.1f}), '
              f'Δ={improvement:.1f}%')

# 2. Does more width help at fixed depth?
print('\n2. WIDTH SCALING (does more d_model help at fixed n_core?)')
for nc in sorted(df['n_core'].unique()):
    sub = df[df['n_core'] == nc].sort_values('d_model')
    if len(sub) >= 2:
        best = sub.loc[sub['final_val_ppl'].idxmin()]
        worst = sub.loc[sub['final_val_ppl'].idxmax()]
        print(f'  n_core={int(nc)}: best d={int(best["d_model"])} (PPL {best["final_val_ppl"]:.1f}), '
              f'worst d={int(worst["d_model"])} (PPL {worst["final_val_ppl"]:.1f})')

# 3. Optimal aspect ratio
print('\n3. OPTIMAL ASPECT RATIO')
best_overall = df.loc[df['final_val_ppl'].idxmin()]
print(f'  Best config: d={int(best_overall["d_model"])}, n_core={int(best_overall["n_core"])}'
      f' → PPL {best_overall["final_val_ppl"]:.2f} ({best_overall["n_params"]/1e6:.1f}M params)')
print(f'  Aspect ratio (d_model / n_core): {int(best_overall["d_model"]) / int(best_overall["n_core"]):.0f}')

# 4. Diminishing returns on depth
print('\n4. DEPTH DIMINISHING RETURNS')
for d in sorted(df['d_model'].unique()):
    sub = df[df['d_model'] == d].sort_values('n_core')
    if len(sub) >= 3:
        ppls = sub['final_val_ppl'].values
        cores = sub['n_core'].values
        gains = []
        for i in range(1, len(ppls)):
            gain = ppls[i-1] - ppls[i]
            gains.append(f'c{int(cores[i-1])}→c{int(cores[i])}: {gain:+.1f}')
        print(f'  d={int(d):>5}: {", ".join(gains)}')

print('\n' + '='*60)

## Parcae Exponent Calculation

Pulls training curves from wandb and computes:
1. **Depth decay rate z** at each token checkpoint
2. **Compute-optimal depth γ** — `depth_opt ∝ tokens^γ` (Parcae: γ=0.40)
3. **Token scaling exponent α** — `L ∝ tokens^(-α)` (Parcae: ~0.78, Chinchilla: ~0.50)

In [ ]:
# ─── Parcae Exponent: optimal depth vs tokens ────────────────────────────────
import wandb
import numpy as np
from scipy.optimize import curve_fit

# wandb already logged in from setup cell

api = wandb.Api()
runs = api.runs("looped-blockell-scaling", filters={"state": "finished"})

# Collect eval PPL at each step for every run
run_curves = {}
for r in runs:
    cfg = r.config
    d_model = cfg.get("d_model")
    n_core  = cfg.get("n_core")
    if d_model is None or n_core is None:
        continue
    
    hist = r.history(keys=["val/ppl", "_step"], samples=10000)
    hist = hist.dropna(subset=["val/ppl"]).sort_values("_step")
    
    if len(hist) == 0:
        continue
    
    key = (int(d_model), int(n_core))
    eff_depth = 1 + n_core * 4 + 1
    tokens_per_step = 32 * 1024

    run_curves[key] = {
        "steps": hist["_step"].values,
        "tokens": hist["_step"].values * tokens_per_step,
        "val_ppl": hist["val/ppl"].values,
        "eff_depth": eff_depth,
    }

print(f"Loaded {len(run_curves)} run curves from wandb")
d_models = sorted(set(k[0] for k in run_curves.keys()))
n_cores  = sorted(set(k[1] for k in run_curves.keys()))
print(f"Widths: {d_models}, Cores: {n_cores}")

# ═══ 1. Depth decay rate z at each token checkpoint ═══════════════════════════
print("\n" + "="*70)
print("1. DEPTH SATURATION RATE: L(depth) = L_inf + Z*exp(-z*depth)")
print("="*70)

eval_steps = np.arange(2500, 77001, 2500)

def exp_model(T, L_inf, Z, z):
    return L_inf + Z * np.exp(-z * T)

for d in d_models:
    cores_for_d = sorted([nc for (dm, nc) in run_curves if dm == d])
    if len(cores_for_d) < 3:
        print(f"\nd={d}: only {len(cores_for_d)} core values, need >=3")
        continue
    
    print(f"\nd={d} (cores: {cores_for_d}):")
    print(f"  {'Tokens':>10} | {'z':>7} | {'L_inf':>7} | {'half-life':>10}")
    print(f"  {'-'*45}")
    
    for step in eval_steps[::4]:  # every 10k steps
        tokens = step * 32768
        depths, ppls = [], []
        
        for nc in cores_for_d:
            key = (d, nc)
            if key not in run_curves:
                continue
            curve = run_curves[key]
            idx = np.argmin(np.abs(curve["steps"] - step))
            if abs(curve["steps"][idx] - step) < 3000:
                depths.append(curve["eff_depth"])
                ppls.append(curve["val_ppl"][idx])
        
        if len(depths) < 3:
            continue
        
        depths = np.array(depths, dtype=float)
        ppls = np.array(ppls, dtype=float)
        
        try:
            p0 = [min(ppls) - 1, max(ppls) - min(ppls) + 5, 0.05]
            popt, _ = curve_fit(exp_model, depths, ppls, p0=p0, maxfev=5000)
            L_inf, Z, z = popt
            if 0 < z < 1 and L_inf > 0:
                hl = np.log(2) / z
                print(f"  {tokens/1e9:>8.2f}B | {z:>7.4f} | {L_inf:>7.1f} | {hl:>8.1f} layers")
        except:
            pass

# ═══ 2. Compute-optimal depth γ ═══════════════════════════════════════════════
print("\n" + "="*70)
print("2. COMPUTE-OPTIMAL DEPTH: depth_opt ∝ tokens^γ  (Parcae γ=0.40)")
print("="*70)

for d in d_models:
    cores_for_d = sorted([nc for (dm, nc) in run_curves if dm == d])
    if len(cores_for_d) < 3:
        continue
    
    opt_depths, opt_tokens = [], []
    
    for step in eval_steps:
        tokens = step * 32768
        best_ppl, best_depth = float('inf'), None
        
        for nc in cores_for_d:
            key = (d, nc)
            if key not in run_curves:
                continue
            curve = run_curves[key]
            idx = np.argmin(np.abs(curve["steps"] - step))
            if abs(curve["steps"][idx] - step) < 3000:
                ppl = curve["val_ppl"][idx]
                if ppl < best_ppl:
                    best_ppl = ppl
                    best_depth = curve["eff_depth"]
        
        if best_depth is not None:
            opt_depths.append(best_depth)
            opt_tokens.append(tokens)
    
    opt_depths = np.array(opt_depths, dtype=float)
    opt_tokens = np.array(opt_tokens, dtype=float)
    
    # Count how often each depth is optimal
    unique, counts = np.unique(opt_depths, return_counts=True)
    print(f"\nd={d}: optimal depth distribution:")
    for u, c in zip(unique, counts):
        nc_approx = (u - 2) / 4
        print(f"  eff_depth={int(u)} (n_core≈{int(nc_approx)}): optimal at {c}/{len(opt_depths)} checkpoints ({c/len(opt_depths)*100:.0f}%)")
    
    # Only fit if there's variation
    if len(unique) >= 2:
        try:
            def power_law(t, a, gamma):
                return a * t**gamma
            popt, _ = curve_fit(power_law, opt_tokens, opt_depths, p0=[1.0, 0.1], maxfev=5000)
            a, gamma = popt
            print(f"  → γ = {gamma:.4f}  (Parcae: 0.40)")
        except Exception as e:
            print(f"  → Power law fit failed: {e}")
    else:
        print(f"  → Same depth optimal throughout (no variation to fit)")

# ═══ 3. Token scaling exponent α ═══════════════════════════════════════════════
print("\n" + "="*70)
print("3. TOKEN SCALING: PPL ∝ tokens^(-α)  (Parcae: ~0.78, Chinchilla: ~0.50)")
print("="*70)

print(f"\n  {'Config':>12} | {'α':>7} | {'floor':>7} | {'RMSE':>6}")
print(f"  {'-'*45}")

all_alphas = []
for d in sorted(d_models):
    for nc in sorted(n_cores):
        key = (d, nc)
        if key not in run_curves:
            continue
        curve = run_curves[key]
        
        tokens = curve["tokens"].astype(float)
        ppls = curve["val_ppl"].astype(float)
        
        if len(tokens) < 5:
            continue
        
        try:
            def token_scaling(t, a, alpha, c):
                return a * t**(-alpha) + c
            
            p0 = [ppls[0] * tokens[0]**0.1, 0.1, min(ppls) * 0.9]
            popt, _ = curve_fit(token_scaling, tokens, ppls, p0=p0, maxfev=10000,
                               bounds=([0, 0.001, 0], [1e20, 2, 1e5]))
            a, alpha, c = popt
            
            pred = token_scaling(tokens, *popt)
            rmse = np.sqrt(np.mean((ppls - pred)**2))
            
            print(f"  d{d:>4}_c{nc} | {alpha:>7.4f} | {c:>7.1f} | {rmse:>6.2f}")
            all_alphas.append(alpha)
        except:
            print(f"  d{d:>4}_c{nc} | {'FAIL':>7} |")

if all_alphas:
    print(f"\n  Mean α = {np.mean(all_alphas):.4f} ± {np.std(all_alphas):.4f}")
    print(f"  Parcae: ~0.78, Chinchilla: ~0.50")
    if np.mean(all_alphas) > 0.6:
        print(f"  → We are in the Parcae regime (efficient token utilization)")
    elif np.mean(all_alphas) > 0.4:
        print(f"  → We are between Chinchilla and Parcae")
    else:
        print(f"  → We are below Chinchilla (undertrained or architecture bottleneck)")
